In [24]:
## Reference
# https://www.coursera.org/learn/introduction-computer-vision-watson-opencv

In [25]:
import copy
import os,sys
os.chdir('/Users/levgolod/Projects/car_classifier/')
from _02_preprocess_data.preprocess_data_fns import *

In [26]:
# importing libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator


import pandas as pd
# import tensorflow.compat.v1 as tf
from common_params import parent_directory_images, parent_directory_url_csvs
import importlib
import sys,os
import pandas as pd
from timethis import timethis



In [27]:
output_filename = f'{parent_directory_url_csvs}df_clip_pred_bodystyle.csv'

In [28]:
IMAGE_SIZE = 256 # increase later
SAMPLE_SIZE = 10000 # increase later

class_names_use = ['sedan','suv','truck']
label_dict = {v:k for k,v in body_style_dict.items()}
print(label_dict)
class_names = list(label_dict.values())
class_labels_use = list([v for k,v in body_style_dict.items() if k in class_names_use])
class_labels_use

{0: 'sedan', 1: 'sportscar', 2: 'suv', 3: 'truck', 4: 'van', 5: 'wagon'}


[0, 2, 3]

### Ingest image data from disk

In [29]:
output_filename_train_val_holdout = f'{parent_directory_url_csvs}df_train_val_hold.csv'
df=pd.read_csv(output_filename_train_val_holdout).sample(frac=1)
df=df.loc[df['body_style'].isin(class_names_use), ].reset_index()
print(df.shape)
print(df.partition.value_counts(dropna=False,normalize=True))
df.head()

(7768, 10)
partition
train    0.718589
hold     0.147915
val      0.133496
Name: proportion, dtype: float64


,index,filepath,filename,filepath_to_use,vehicle_id,partition,make,model,body_style,body_style_index
0,7774,/Users/levgolod/Projects/car_classifier/data/a...,b30102c7073244ef89ad42ac255012c9.jpg,/Users/levgolod/Projects/car_classifier/data/a...,733252905,hold,toyota,tacoma,truck,3
1,3359,/Users/levgolod/Projects/car_classifier/data/a...,d4d53209298f48869bfa26a4b6b7381b.jpg,/Users/levgolod/Projects/car_classifier/data/a...,738029080,train,ford,f150,truck,3
2,888,/Users/levgolod/Projects/car_classifier/data/a...,4cc2c0f8e8984fd49e5e7b76ee72567e.jpg,/Users/levgolod/Projects/car_classifier/data/a...,738678846,val,bmw,4-series,sedan,0
3,3048,/Users/levgolod/Projects/car_classifier/data/a...,d93e9677fae545f1831e83cd35554772.jpg,/Users/levgolod/Projects/car_classifier/data/a...,735399632,train,ford,f150,truck,3
4,7975,/Users/levgolod/Projects/car_classifier/data/a...,d67b6914e8784fd699cd1b5834750cb6.jpg,/Users/levgolod/Projects/car_classifier/data/a...,737772330,hold,toyota,tacoma,truck,3


In [30]:
if False:

    train_dataset = process_image_files(
        df.loc[df['partition']=='train', ['filepath_to_use','body_style_index']].head(SAMPLE_SIZE).values,
        image_size=IMAGE_SIZE,
    )

    test_dataset = process_image_files(
        df.loc[df['partition']=='val', ['filepath_to_use','body_style_index']].head(SAMPLE_SIZE).values,
        image_size=IMAGE_SIZE,
    )

    holdout_dataset = process_image_files(
        df.loc[df['partition']=='hold', ['filepath_to_use','body_style_index']].head(SAMPLE_SIZE).values,
        image_size=IMAGE_SIZE,
    )

    print(len(train_dataset),len(test_dataset),len(holdout_dataset))
    train_images, train_labels = \
    ptrain_images, train_labels = \
        [x[0] for x in train_dataset], [x[1] for x in train_dataset]

    test_images, test_labels = \
        [x[0] for x in test_dataset], [x[1] for x in test_dataset]

    holdout_images, holdout_labels = \
        [x[0] for x in holdout_dataset], [x[1] for x in holdout_dataset]

    ## coerce to array - data
    train_images = np.array([np.array(x) for x in train_images])
    test_images = np.array([np.array(x) for x in test_images])
    holdout_images = np.array([np.array(x) for x in holdout_images])

    ## coerce to array - labels
    train_labels = np.array(train_labels)
    test_labels = np.array(test_labels)
    holdout_labels = np.array(holdout_labels)

In [31]:
class_names_use
i=0
record=dict(df.loc[i, ])
record

probs = get_predicted_categories_clip(
            image_path= record['filepath'],
            categories=class_names_use
        )
probs

top_category  = probs.head(1).index[0]
probs_dict = dict(probs)
top_category, probs_dict

('truck', {'truck': 0.7247826, 'suv': 0.27518618, 'sedan': 3.1257317e-05})

In [ ]:
df['clip_probs']=''
df['clip_prediction']=''

iterprint = 100
for i,record in df.iterrows():
    if i % iterprint==0:
        message =\
f'''###########################################################
### Begin processing {i} of {len(df)} | {dt.datetime.now().replace(microsecond=0)}
###########################################################'''
        print(message) # second

    image_path= record['filepath']
    try:
        probs = get_predicted_categories_clip(
                image_path,
                categories=class_names_use
            )
        top_category  = probs.head(1).index[0]
        probs_dict = dict(probs)
        df.loc[i, 'clip_probs'] = str(probs_dict)
        df.loc[i, 'clip_prediction'] = str(top_category)
    except Exception as e:
        print(f'\nProblem with {i} {image_path}')
        print(e)





###########################################################
### Begin processing 0 of 7768 | 2025-03-03 11:19:26
###########################################################


In [20]:
df.head()



,index,filepath,filename,filepath_to_use,vehicle_id,partition,make,model,body_style,body_style_index,clip_probs,clip_prediction
0,5592,/Users/levgolod/Projects/car_classifier/data/a...,30e0e9517e0a4cf89b6d58784dfba9cb.jpg,/Users/levgolod/Projects/car_classifier/data/a...,730103361,train,honda,passport,suv,2,"{'suv': 0.946043, 'truck': 0.028780062, 'sedan...",suv
1,4308,/Users/levgolod/Projects/car_classifier/data/a...,0b5c66270e394dee9cbfcf729e2031ae.jpg,/Users/levgolod/Projects/car_classifier/data/a...,739401021,train,ford,taurus,sedan,0,"{'suv': 0.91179085, 'sedan': 0.084755264, 'tru...",suv
2,3719,/Users/levgolod/Projects/car_classifier/data/a...,571084edc3f5470381fc1e430a76f7d7.jpg,/Users/levgolod/Projects/car_classifier/data/a...,444880994,train,ford,taurus,sedan,0,"{'sedan': 0.6506132, 'suv': 0.3464937, 'truck'...",sedan
3,798,/Users/levgolod/Projects/car_classifier/data/a...,6337022a29e14ab59c427230bc5ab69d.jpg,/Users/levgolod/Projects/car_classifier/data/a...,737155770,train,bmw,4-series,sedan,0,"{'suv': 0.85583186, 'sedan': 0.138541, 'truck'...",suv
4,5122,/Users/levgolod/Projects/car_classifier/data/a...,2d4db4895c324452bed3be7bd3dbf41d.jpg,/Users/levgolod/Projects/car_classifier/data/a...,738629751,hold,honda,civic,sedan,0,"{'sedan': 0.8744824, 'suv': 0.1222223, 'truck'...",sedan


In [ ]:
df.to_csv(output_filename,index=False)